# 09. Testing temporal 2026

Training tetap 2021-2025. Model, scaler, threshold dan parameter dibekukan sebelum transformasi data 2026. Tidak ada label ground truth untuk menghitung accuracy.

[Panduan](../00_panduan.ipynb) | [Sebelumnya](../08_stability/08_stability.ipynb) | [Berikutnya](../10_gis_mapping/10_gis_mapping.ipynb)

### Daftar bagian

- [1. Inisialisasi dan dependensi](#s1)
- [2. Aturan ekstensi DBSCAN](#s2)
- [3. Prediksi K-Means, drift, dan periode parsial](#s3)
- [4. Jalankan testing 2026](#s4)
- [5. Audit kecocokan dan quarter](#s5)
- [6. Baca hasil testing dengan nama kelompok historis](#s6)

<a id="s1"></a>
## 1. Inisialisasi dan dependensi

Jalankan dari atas ke bawah dengan kernel Python 3.12.11 (Seismic Research). Konfigurasi tersentral di config.json.

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")
candidates = []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "seismic_research_2021_2025"])
ROOT = next((p for p in candidates if (p / "config.json").is_file() and (p / "src/common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari folder proyek seismic_research_2021_2025.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Image, Markdown, FileLink
from src.common import read_config
cfg = read_config(ROOT)
print("Interpreter:", sys.executable)
print("Python:", sys.version.split()[0])
print("Proyek:", ROOT)
print("Training:", cfg["train_start"], "sampai sebelum", cfg["train_end_exclusive"])
print("Testing sampai sebelum (UTC):", cfg["test_end_exclusive"])

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from src.common import ROOT, load_model, load_prepared, write_json
from src.modeling import cluster_metrics
from src.preprocessing import grid_panel

Interpreter: C:\Users\yudhi\.local\bin\python3.12.exe
Python: 3.12.11
Proyek: C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025
Training: 2021-01-01 sampai sebelum 2026-01-01
Testing sampai sebelum (UTC): 2026-10-26


<a id="s2"></a>
## 2. Aturan ekstensi DBSCAN

Scikit-learn DBSCAN tidak menyediakan predict bawaan. Ekstensi ini menugaskan titik hanya jika core points dalam radius eps berasal dari satu cluster. Tidak ada kecocokan -> NO_CORE_MATCH; beberapa cluster -> AMBIGUOUS_CORE_MATCH. Ini aturan tambahan yang dapat ditolak, bukan refit DBSCAN pada 2026.

In [2]:
def extend_dbscan(model, X):
    labels = np.full(len(X), -1, dtype=int)
    matching_clusters = np.zeros(len(X), dtype=int)
    distances = np.full(len(X), np.nan)
    if len(X) == 0 or len(model.components_) == 0:
        return labels, matching_clusters, distances
    core_labels = model.labels_[model.core_sample_indices_]
    nn = NearestNeighbors(radius=model.eps, n_jobs=1).fit(model.components_)
    nearest_distances, _ = nn.kneighbors(X, n_neighbors=1)
    distances = nearest_distances[:,0]
    # Per batch agar radius-neighbor tidak menampung seluruh daftar sekaligus.
    for start in range(0, len(X), 500):
        neighbors = nn.radius_neighbors(X[start:start+500], return_distance=False)
        for offset, indices in enumerate(neighbors):
            clusters = np.unique(core_labels[indices])
            row = start + offset
            matching_clusters[row] = len(clusters)
            if len(clusters) == 1:
                labels[row] = clusters[0]
            elif len(clusters) > 1:
                labels[row] = -2
    return labels, matching_clusters, distances

<a id="s3"></a>
## 3. Prediksi K-Means, drift, dan periode parsial

K-Means selalu memberi centroid terdekat; jarak melampaui kuantil 99% training ditandai sebagai flag eksploratif, bukan uji signifikansi. Q1-Q3 2026 dilaporkan terpisah dari Q4 parsial. Frekuensi Q4 dinormalisasi lama pengamatan, tetapi statistik magnitudo dan maksimum tetap tidak sebanding penuh dengan triwulan lengkap. Pergantian dibanding baseline 2025Q4, bukan otomatis pergantian dari triwulan tepat sebelumnya.

In [3]:
def testing_2026(root=ROOT):
    root = Path(root)
    cfg, panel, _, Xtrain, bundle, _ = load_prepared(root)
    km, _ = load_model(root, "kmeans")
    db, dbmeta = load_model(root, "dbscan")
    test = panel.loc[panel["split"].ne("train")].copy()
    eligible = test.loc[test["state"].eq("ELIGIBLE")].copy()
    if eligible.empty:
        raise ValueError("Tidak ada observasi 2026 yang layak.")
    X = bundle["scaler"].transform(eligible[cfg["features"]])
    with threadpool_limits(limits=2):
        km_labels = km.predict(X)
        distances = km.transform(X).min(axis=1)
        training_distances = km.transform(Xtrain).min(axis=1)
        threshold = float(np.quantile(training_distances, .99))
        db_labels, matches, core_distances = extend_dbscan(db, X)
    predicted = eligible[["row_id"]].copy()
    predicted["kmeans_cluster"] = km_labels
    predicted["kmeans_profile"] = [f"KMEANS_P{label}" for label in km_labels]
    predicted["centroid_distance"] = distances
    predicted["outside_training_distance_q99"] = distances > threshold
    predicted["dbscan_cluster"] = db_labels
    predicted["dbscan_profile"] = ["AMBIGUOUS_CORE_MATCH" if x == -2 else "NO_CORE_MATCH" if x == -1 else f"DBSCAN_P{x}" for x in db_labels]
    predicted["matching_core_clusters"] = matches
    predicted["nearest_core_distance"] = core_distances
    test = test.merge(predicted, on="row_id", how="left", validate="one_to_one")
    for method in ["kmeans", "dbscan"]:
        test[f"{method}_profile"] = test[f"{method}_profile"].fillna(test["state"])
    out = root / "reports/testing_2026"
    test.to_csv(out / "labels_all_2026.csv", index=False)
    test.loc[test["split"].eq("test")].to_csv(out / "labels_complete_quarters.csv", index=False)
    test.loc[test["split"].eq("test_partial")].to_csv(out / "labels_partial_quarter.csv", index=False)
    rows = []
    for split in ["test", "test_partial"]:
        mask = eligible["split"].eq(split).to_numpy()
        if not mask.any():
            continue
        for method, labels in [("kmeans", km_labels), ("dbscan_core_extension", db_labels)]:
            rows.append({"split": split, "method": method, "ambiguous_fraction": float((labels[mask] == -2).mean()),
                         **cluster_metrics(X[mask], labels[mask], cfg["silhouette_sample_size"])})
    metrics = pd.DataFrame(rows)
    metrics.to_csv(out / "holdout_metrics.csv", index=False)
    drift = []
    train = panel.loc[panel["split"].eq("train") & panel["state"].eq("ELIGIBLE")]
    complete = eligible.loc[eligible["split"].eq("test")]
    for feature in cfg["features"]:
        sd = train[feature].std(ddof=0)
        drift.append({"feature": feature, "train_mean": train[feature].mean(), "test_complete_mean": complete[feature].mean(),
                      "standardized_mean_shift": (complete[feature].mean()-train[feature].mean())/sd if sd else np.nan})
    pd.DataFrame(drift).to_csv(out / "feature_drift.csv", index=False)
    changes = []
    for method in ["kmeans", "dbscan"]:
        last = pd.read_csv(root / f"models/{method}/training_labels.csv")
        last = last.loc[last["period"].eq("2025Q4"), ["grid_id", "profile"]].rename(columns={"profile": "previous_profile"})
        combined = test.merge(last, on="grid_id", how="left")
        combined["model"] = method
        combined["profile"] = combined[f"{method}_profile"]
        combined["baseline_period"] = "2025Q4"
        combined["comparable_to_baseline"] = combined["profile"].str.contains("_P", regex=False) & combined["previous_profile"].fillna("").str.contains("_P", regex=False)
        combined["changed_from_baseline"] = combined["comparable_to_baseline"] & combined["profile"].ne(combined["previous_profile"])
        changes.append(combined[["grid_id", "period", "split", "model", "baseline_period", "previous_profile", "profile", "comparable_to_baseline", "changed_from_baseline"]])
    pd.concat(changes).to_csv(out / "changes_from_2025Q4.csv", index=False)
    write_json(out / "methodology.json", {"train_period": "2021Q1-2025Q4", "test_end_exclusive_utc": cfg["test_end_exclusive"],
               "complete_quarters": sorted(test.loc[test["split"].eq("test"), "period"].unique().tolist()),
               "partial_quarters": sorted(test.loc[test["split"].eq("test_partial"), "period"].unique().tolist()),
               "kmeans_distance_q99_training": threshold, "dbscan_method": "core-radius extension, bukan predict bawaan; beberapa cluster core dalam eps = ambigu",
               "dbscan_diagnostic_only": dbmeta["diagnostic_only"], "accuracy": "tidak dihitung karena tidak ada label ground truth"})
    return metrics, test

<a id="s4"></a>
## 4. Jalankan testing 2026



In [4]:
holdout_metrics, test_labels = testing_2026(ROOT)
display(holdout_metrics)
display(test_labels.groupby(["period","split","kmeans_profile"]).size().rename("n").reset_index())
display(pd.read_csv(ROOT / "reports/testing_2026/feature_drift.csv"))

,split,method,ambiguous_fraction,n_rows,n_clusters,n_assigned,noise_fraction,largest_cluster_fraction_assigned,smallest_cluster_size,silhouette,davies_bouldin,silhouette_n
0,test,kmeans,0.000000,2625,6,2625,0.000000,0.468952,91,0.376161,0.973501,2000
1,test,dbscan_core_extension,0.000381,2625,7,2277,0.132571,0.433904,5,0.155648,2.013470,2000
2,test_partial,kmeans,0.000000,298,6,298,0.000000,0.355705,8,0.397524,0.870319,298
3,test_partial,dbscan_core_extension,0.000000,298,2,172,0.422819,0.953488,8,0.585860,0.455596,172


,period,split,kmeans_profile,n
0,2026Q1,test,KMEANS_P0,122
1,2026Q1,test,KMEANS_P1,32
2,2026Q1,test,KMEANS_P2,126
3,2026Q1,test,KMEANS_P3,56
4,2026Q1,test,KMEANS_P4,438
5,2026Q1,test,KMEANS_P5,153
6,2026Q1,test,NO_EVENTS,912
7,2026Q2,test,KMEANS_P0,85
8,2026Q2,test,KMEANS_P1,33
9,2026Q2,test,KMEANS_P2,132


,feature,train_mean,test_complete_mean,standardized_mean_shift
0,log_frequency,0.543130,0.646476,0.281140
1,avg_magnitude,3.710893,3.538019,-0.354561
2,max_magnitude,3.955408,3.831619,-0.194914
3,avg_depth,65.630711,63.747280,-0.018422
4,shallow_fraction,0.738257,0.731714,-0.016193


<a id="s5"></a>
## 5. Audit kecocokan dan quarter



In [5]:
display(test_labels.groupby(["period","dbscan_profile"]).size().rename("n").reset_index())
assert set(test_labels.loc[test_labels["split"].eq("test"),"period"]) == {"2026Q1","2026Q2","2026Q3"}
print("Q4 disimpan pada labels_partial_quarter.csv; tidak digabung ke evaluasi quarter lengkap.")

,period,dbscan_profile,n
0,2026Q1,AMBIGUOUS_CORE_MATCH,1
1,2026Q1,DBSCAN_P0,284
2,2026Q1,DBSCAN_P1,29
3,2026Q1,DBSCAN_P2,93
4,2026Q1,DBSCAN_P3,353
5,2026Q1,DBSCAN_P4,36
6,2026Q1,DBSCAN_P5,11
7,2026Q1,DBSCAN_P6,3
8,2026Q1,NO_CORE_MATCH,117
9,2026Q1,NO_EVENTS,912


Q4 disimpan pada labels_partial_quarter.csv; tidak digabung ke evaluasi quarter lengkap.


<a id="s6"></a>
## 6. Baca hasil testing dengan nama kelompok historis

Nama berasal dari ciri kelompok training yang dijelaskan pada notebook 04/05. Data 2026 hanya dicocokkan dengan kelompok historis; angka pada baris di bawah adalah jumlah kotak, bukan jumlah event gempa.

In [6]:
from src.mapping import profile_descriptions
for method in ["kmeans","dbscan"]:
    descriptions = profile_descriptions(ROOT, method, cfg)
    named_test = test_labels.copy()
    named_test["nama_kelompok"] = named_test[f"{method}_profile"].map(lambda code: descriptions[code]["name"])
    display(Markdown(f"### Hasil pengujian {method.upper()}"))
    with pd.option_context("display.max_colwidth", None):
        display(named_test.groupby(["period","split","nama_kelompok"]).size().rename("jumlah_kotak").reset_index())

### Hasil pengujian KMEANS

,period,split,nama_kelompok,jumlah_kotak
0,2026Q1,test,"Kelompok A: Cenderung dangkal, magnitudo sekitar 4,2",122
1,2026Q1,test,"Kelompok B: Kedalaman sekitar 468 km, magnitudo 4,2",32
2,2026Q1,test,"Kelompok C: Kedalaman sekitar 138 km, magnitudo 3,4",126
3,2026Q1,test,"Kelompok D: Kedalaman sekitar 145 km, magnitudo 4,2",56
4,2026Q1,test,"Kelompok E: Cenderung dangkal, magnitudo sekitar 3,4",438
5,2026Q1,test,Kelompok F: Kejadian lebih sering; cenderung dangkal,153
6,2026Q1,test,Tidak ada gempa tercatat dengan magnitudo minimal 3,912
7,2026Q2,test,"Kelompok A: Cenderung dangkal, magnitudo sekitar 4,2",85
8,2026Q2,test,"Kelompok B: Kedalaman sekitar 468 km, magnitudo 4,2",33
9,2026Q2,test,"Kelompok C: Kedalaman sekitar 138 km, magnitudo 3,4",132


### Hasil pengujian DBSCAN

,period,split,nama_kelompok,jumlah_kotak
0,2026Q1,test,Belum cocok dengan kelompok historis,117
1,2026Q1,test,Cocok dengan lebih dari satu kelompok,1
2,2026Q1,test,"Kelompok A: Cenderung dangkal, magnitudo sekitar 3,7",284
3,2026Q1,test,"Kelompok B: Kedalaman sekitar 144 km, magnitudo 3,9",29
4,2026Q1,test,"Kelompok C: Kedalaman sekitar 209 km, magnitudo 3,9",93
5,2026Q1,test,Kelompok D: Kejadian lebih sering; cenderung dangkal,353
6,2026Q1,test,"Kelompok E: Kedalaman sekitar 75 km, magnitudo 3,7",36
7,2026Q1,test,"Kelompok F: Kedalaman sekitar 95 km, magnitudo 3,6",11
8,2026Q1,test,"Kelompok G: Kedalaman sekitar 285 km, magnitudo 4,2",3
9,2026Q1,test,Tidak ada gempa tercatat dengan magnitudo minimal 3,912


## Referensi dan batas interpretasi

[Dokumentasi EMSC](https://www.seismicportal.eu/fdsn-wsevent.html), [clustering scikit-learn](https://scikit-learn.org/stable/modules/clustering.html), dan docs/metodologi.md. Hasil adalah profil aktivitas katalog dalam bounding box dan periode yang dipilih. Interpretasi risiko, prediksi kejadian gempa, serta klaim perubahan fisik memerlukan bukti tambahan.